# Hyperion Economy — Juno Story Capsule

Ein präsentationsfertiger Ablauf für den **Agentenrat**, den **Scenario Composer**, die **Time-Debt-Routenkarte**, den **AI Flight Recorder** und den reproduzierbaren Export.

In [ ]:
import pandas as pd
from IPython.display import display
from juno_showcase import (
    BOKEH_AVAILABLE, DEMO_MODES, agent_council_table, bokeh_agent_dashboard,
    bokeh_route_map, bokeh_scenario_dashboard, bokeh_status,
    build_showcase, compose_scenario, configure_bokeh_notebook, demo_mode_config,
    export_demo_capsule,
    flight_recorder_table, scenario_comparison,
)

print(bokeh_status())
if BOKEH_AVAILABLE:
    configure_bokeh_notebook()

DEMO_MODE = 'einsteiger'
MODE_CONFIG = demo_mode_config(DEMO_MODE)
EPISODES = MODE_CONFIG['episodes']
YEARS = MODE_CONFIG['years']
SEED = 7

## 0. Demo-Modus waehlen

### Zweck der Auswahl

Das Profil steuert Rechenzeit, sichtbare Abschnitte und Exportumfang. So bleibt dieselbe Simulation fuer eine kurze Einfuehrung lesbar und kann fuer eine technische Analyse schrittweise geoeffnet werden.

- `einsteiger`: Agentenrat und drei Szenarien.
- `agentenvergleich`: zusaetzlich Lernkurven, Endwertvergleich und kurze Entscheidungslogs.
- `tiefenanalyse`: zusaetzlich Routenkarte, vollstaendiger Flight Recorder und Demo-Capsule.

Nach einer Dropdown-Auswahl die Zellen ab **Der Agentenrat** erneut ausfuehren.
Die Auswahl setzt Laufzeit, sichtbare Abschnitte und Exportumfang. Ohne Widgets kann `DEMO_MODE` in der vorherigen Zelle geaendert werden. Nach einer neuen Auswahl die Zellen ab **Der Agentenrat** erneut ausfuehren.

In [ ]:
try:
    import ipywidgets as widgets
    def select_demo_mode(mode='einsteiger'):
        global DEMO_MODE, MODE_CONFIG, EPISODES, YEARS
        DEMO_MODE = mode
        MODE_CONFIG = demo_mode_config(mode)
        EPISODES = MODE_CONFIG['episodes']
        YEARS = MODE_CONFIG['years']
        display(pd.DataFrame([{
            'Modus': MODE_CONFIG['label'],
            'Episoden': EPISODES,
            'Jahre': YEARS,
            'Routenkarte': MODE_CONFIG['show_route_map'],
            'Flight Recorder': MODE_CONFIG['show_flight_recorder'],
            'Capsule-Export': MODE_CONFIG['export_capsule'],
        }]))
        print('Auswahl uebernommen; ab Der Agentenrat erneut ausfuehren.')
    display(widgets.interact(
        select_demo_mode,
        mode=widgets.Dropdown(
            options=[(config['label'], key) for key, config in DEMO_MODES.items()],
            value=DEMO_MODE, description='Demo-Modus'
        ),
    ))
except ImportError:
    print('ipywidgets ist optional; DEMO_MODE in der ersten Zelle verwenden.')

## 1. Der Agentenrat

### So liest du die Tabelle

Die drei Agenten sehen dieselben Marktbedingungen, verfolgen aber unterschiedliche Lernziele. `avg_final_value` ist der durchschnittliche Portfolio-Endwert, `win_rate` der Anteil der gewonnenen Buy-and-Hold-Vergleiche. `liquidity_stress_rate` und `insolvency_rate` zeigen, ob eine scheinbar profitable Strategie unter Druck geriet.

Die Werte sind Vergleichswerte innerhalb dieser Simulation, keine Prognosen fuer reale Maerkte.
Die drei Rollen lernen auf denselben Hyperion-Marktverläufen, bleiben aber als Strategie vergleichbar.

In [ ]:
showcase = build_showcase(episodes=EPISODES, years=YEARS, seed=SEED)
council = agent_council_table(showcase['summary'])
display(council.round(2))

In [ ]:
if BOKEH_AVAILABLE and MODE_CONFIG['show_agent_dashboard']:
    from bokeh.io import show
    show(bokeh_agent_dashboard(showcase))
else:
    display(showcase['summary'].round(2))

## 2. Scenario Composer

### Was wird verglichen?

Ein Zwangsevent macht den Eintrittszeitpunkt kontrollierbar. `final_stability`, `final_prosperity`, `total_trade_value` und `final_time_debt` zeigen gemeinsam, ob ein Schock nur kurzfristig stoert oder langfristige Kosten erzeugt. Fuer belastbare Aussagen sollten mehrere Seeds und Ereignisjahre getestet werden.
Das Ereignis und sein Eintrittsjahr können für eine Live-Demo verändert werden.

In [ ]:
scenario_frame = scenario_comparison(
    event_names=MODE_CONFIG['events'], years=YEARS, seed=SEED,
    event_year=MODE_CONFIG['event_year'],
)
display(scenario_frame.round(2))
if BOKEH_AVAILABLE and MODE_CONFIG['show_scenario_dashboard']:
    show(bokeh_scenario_dashboard(scenario_frame))

In [ ]:
try:
    import ipywidgets as widgets
    def choose_event(event_name='farcaster_stoerung', event_year=3):
        _, kpi = compose_scenario(event_name=event_name, event_year=event_year, years=YEARS, seed=SEED)
        display(kpi.round(2))
    display(widgets.interact(
        choose_event,
        event_name=widgets.Dropdown(
            options=['keines', 'farcaster_stoerung', 'ouster_raid', 'pilgerboom', 'sanktionen'],
            value='farcaster_stoerung', description='Event'
        ),
        event_year=widgets.IntSlider(value=MODE_CONFIG['event_year'], min=1, max=YEARS, step=1, description='Jahr'),
    ))
except ImportError:
    print('ipywidgets ist optional; die feste Scenario-Tabelle bleibt verfügbar.')

## 3. Time-Debt-Routenkarte

### Lesen der schematischen Karte

Die x-Achse ordnet Welten nach Kern, Grenze und Peripherie; sie ist keine geografische Projektion. Linien stammen aus echten Handelsrecords. Tooltips verbinden Handelsvolumen und Transportkosten mit Stabilitaet und Time Debt am Zielknoten.
Die Karte ist schematisch: Positionen zeigen Kern, Grenze und Peripherie. Linien stammen aus echten Handelsrecords; Farbe und Tooltip erklären Transportkosten und Ziel-Time-Debt.

In [ ]:
if MODE_CONFIG['show_route_map']:
    display(showcase['routes'].head(10))
    if BOKEH_AVAILABLE:
        show(bokeh_route_map(showcase['routes'], showcase['nodes']))
else:
    print('Routenkarte im gewaehlten Modus ausgeblendet.')

## 4. AI Flight Recorder

### Von der Aktion zur Erklaerung

`best_action` ist die aktuell fuehrende Aktion im Lernmodell. Weicht `action` davon ab, war die Entscheidung explorativ. `realized_reward` wird erst nach dem naechsten Marktjahr bekannt, weil Wertveraenderung und Risiko dann abgerechnet sind.
Jede Entscheidung kann aufgerufen werden: Zustand, Aktion, Q-Werte, Exploration, Kosten und realisierte Belohnung.

In [ ]:
if MODE_CONFIG['show_flight_recorder']:
    recorder = flight_recorder_table(showcase['decisions'])
    display(recorder[[
        'year', 'agent', 'action', 'best_action', 'price',
        'event_count', 'time_debt', 'realized_reward', 'explanation'
    ]].head(MODE_CONFIG['recorder_rows']))
else:
    print('Flight Recorder im gewaehlten Modus ausgeblendet.')

## 5. Reproduzierbare Demo Capsule

### Was wird gespeichert?

Die Capsule enthaelt CSV-Dateien fuer Simulation, Agentenrat, Szenarien, Routen und Flight Recorder sowie Bericht und Metadaten. Codeversion, Parameter und Seed gehoeren zusammen; die Capsule ist ein nachvollziehbarer Vorfuehrungsbeleg, kein reales Wirtschaftsprognosemodell.

Wenn Bokeh installiert ist, wird zusaetzlich ein eigenstaendiges HTML-Dashboard exportiert. Ohne Bokeh bleiben Tabellen und CSV voll nutzbar.
Der Export erzeugt Tabellen, einen Markdown-Bericht, Metadaten und – wenn Bokeh verfügbar ist – ein eigenständiges interaktives HTML-Dashboard.

In [ ]:
if MODE_CONFIG['export_capsule']:
    showcase['council_markdown'] = agent_council_table(showcase['summary']).to_string(index=False)
    capsule = export_demo_capsule(showcase, output_dir='demo_capsule')
    print('Capsule geschrieben:')
    display(pd.DataFrame({'file': capsule['files']}))
else:
    print('Capsule-Export im gewaehlten Modus ausgelassen.')

### Präsentationsablauf

1. **3 Minuten:** Agentenrat und Endwertvergleich.
2. **8 Minuten:** Farcaster-Ausfall im Composer vorziehen und Auswirkungen zeigen.
3. **15 Minuten:** Route öffnen, eine Agentenentscheidung im Flight Recorder erklären und die Capsule exportieren.

Alle Daten bleiben lokal in Juno. Für eine stabile Vorführung kann `SEED = 7` unverändert bleiben.

### Sprecherlogik

Erst die Rollen und die Frage nach der robustesten Strategie zeigen. Danach ein Ereignis vorziehen und die Kennzahlen vergleichen. Zum Schluss eine konkrete Entscheidung im Flight Recorder oeffnen und erst dann die technischen Details erklaeren.